# Part B: Customer Churn Prediction
### Machine Learning Project – Classification
---

## 1. Understanding the Problem

**What is Customer Churn?**

Customer churn means a customer **stops using a company's service**. This is a big problem for businesses because getting new customers is much more expensive than keeping existing ones.

**What are we trying to do?**

We will build a **classification model** that predicts **whether a customer will churn (Yes/No)** based on:
- Their demographic info (gender, age, family)
- Services they use (internet, phone, streaming)
- Account details (contract type, monthly charges, tenure)

**Target Variable:** `Churn` → Yes (will leave) or No (will stay)

**Type of Problem:** Binary Classification

## 2. Import Libraries

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries imported successfully!")

## 3. Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_excel('Customer_data.xlsx')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
# View the first few rows
df.head()

In [ ]:
# Check column names and data types
df.info()

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

### 3.1 Exploratory Data Analysis (EDA)

In [ ]:
# How many customers churned vs stayed?
plt.figure(figsize=(6, 5))
churn_counts = df['Churn'].value_counts()
sns.barplot(x=churn_counts.index, y=churn_counts.values, palette=['steelblue', 'coral'])
plt.title('Churn Distribution (Target Variable)', fontsize=14)
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    plt.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

churn_rate = (df['Churn'] == 'Yes').mean() * 100
print(f"Churn Rate: {churn_rate:.1f}%")

In [ ]:
# Churn by Contract Type
plt.figure(figsize=(9, 5))
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
contract_churn.columns = ['Contract', 'Churn Rate (%)']

sns.barplot(data=contract_churn, x='Contract', y='Churn Rate (%)', palette='Set2')
plt.title('Churn Rate by Contract Type', fontsize=14)
plt.ylabel('Churn Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Tenure for Churned vs Non-Churned
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True,
             palette={'No': 'steelblue', 'Yes': 'coral'})
plt.title('Tenure Distribution by Churn Status', fontsize=14)
plt.xlabel('Tenure (Months)')
plt.tight_layout()
plt.show()

print("Average tenure for churned customers:",
      df[df['Churn']=='Yes']['tenure'].mean().round(1), "months")
print("Average tenure for retained customers:",
      df[df['Churn']=='No']['tenure'].mean().round(1), "months")

In [ ]:
# Monthly Charges vs Churn
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges',
            palette={'No': 'steelblue', 'Yes': 'coral'})
plt.title('Monthly Charges by Churn Status', fontsize=14)
plt.xlabel('Churn')
plt.ylabel('Monthly Charges ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by Internet Service type
plt.figure(figsize=(8, 5))
internet_churn = df.groupby('InternetService')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
internet_churn.columns = ['InternetService', 'Churn Rate (%)']

sns.barplot(data=internet_churn, x='InternetService', y='Churn Rate (%)', palette='Oranges')
plt.title('Churn Rate by Internet Service', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Drop the customerID column – it's just an identifier, not useful for prediction
df_model = df.drop('customerID', axis=1).copy()

print(f"Dataset shape after dropping ID: {df_model.shape}")

In [ ]:
# Handle missing values
# TotalCharges has 11 missing values – fill with median
df_model['TotalCharges'].fillna(df_model['TotalCharges'].median(), inplace=True)

print("Missing values after handling:")
print(df_model.isnull().sum().sum(), "total missing values")

In [ ]:
# Identify categorical columns to encode
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:")
print(cat_cols)

In [ ]:
# Encode categorical columns using Label Encoding
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

print("Label encoding complete!")
df_model.head()

In [ ]:
# Separate features (X) and target (y)
X = df_model.drop('Churn', axis=1)   # Features
y = df_model['Churn']                 # Target: 0=No, 1=Yes

print(f"Features shape: {X.shape}")
print(f"Target shape  : {y.shape}")
print(f"Churn (1=Yes) distribution:\n{y.value_counts()}")

In [ ]:
# Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} rows")
print(f"Testing set size : {X_test.shape[0]} rows")

## 5. Model Building

In [ ]:
# --- Model 1: Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

print("Logistic Regression model trained!")

In [ ]:
# --- Model 2: Decision Tree Classifier ---
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)

print("Decision Tree model trained!")

In [ ]:
# --- Model 3: Random Forest Classifier (Best model) ---
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("Random Forest model trained!")

## 6. Model Evaluation

In [ ]:
# Function to compute and print all classification metrics
def evaluate_model(name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    print(f"--- {name} ---")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print()
    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1 Score': f1}

results = []
results.append(evaluate_model('Logistic Regression', y_test, lr_preds))
results.append(evaluate_model('Decision Tree',       y_test, dt_preds))
results.append(evaluate_model('Random Forest',       y_test, rf_preds))

In [ ]:
# Compare models in a table
results_df = pd.DataFrame(results).set_index('Model')
print("Model Comparison Table:")
results_df

In [ ]:
# Bar chart comparing all metrics
results_df.plot(kind='bar', figsize=(12, 5), colormap='Set2', edgecolor='black')
plt.title('Model Performance Comparison', fontsize=14)
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix for the best model (Random Forest)
cm = confusion_matrix(y_test, rf_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed (0)', 'Churned (1)'],
            yticklabels=['Stayed (0)', 'Churned (1)'])
plt.title('Confusion Matrix – Random Forest', fontsize=14)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print("\nHow to read this:")
print(f"  Correctly predicted 'Stayed' : {cm[0][0]}")
print(f"  Wrongly predicted as 'Churned': {cm[0][1]}")
print(f"  Missed churners (False Negative): {cm[1][0]}")
print(f"  Correctly predicted 'Churned' : {cm[1][1]}")

In [ ]:
# Full classification report
print("Detailed Classification Report (Random Forest):")
print(classification_report(y_test, rf_preds,
                            target_names=['Stayed (0)', 'Churned (1)']))

In [ ]:
# Feature Importance from Random Forest
importances = rf_model.feature_importances_
feat_importance = pd.Series(importances, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feat_importance.values, y=feat_importance.index, palette='magma')
plt.title('Feature Importance (Random Forest)', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("Top 5 features driving churn:")
print(feat_importance.head())

In [ ]:
# Sample predictions on test data
sample = X_test.head(10).copy()
sample['Actual Churn']    = y_test.values[:10]
sample['Predicted Churn'] = rf_preds[:10]
sample['Churn Label']     = sample['Predicted Churn'].map({0: 'No', 1: 'Yes'})

print("Sample Predictions (1 = Churned, 0 = Stayed):")
sample[['Actual Churn', 'Predicted Churn', 'Churn Label']].reset_index(drop=True)

## 7. Conclusion

Here is a summary of what we found in this project:

1. **Month-to-month contracts churn the most** – Customers on short-term contracts are far more likely to leave. Offering incentives to switch to annual contracts could reduce churn significantly.

2. **New customers are at higher risk** – Customers with low tenure (less than 12 months) churn more often. This suggests the onboarding experience needs improvement.

3. **Higher monthly charges correlate with churn** – Customers paying more per month tend to leave more. Targeted discounts or value-added services can help retain them.

4. **Random Forest gave the best performance** – It outperformed Logistic Regression and Decision Tree on all metrics, especially Recall (catching actual churners).

5. **Actionable insight** – The company should proactively reach out to customers with: high monthly charges, month-to-month contracts, and tenure under 12 months — these are the highest-risk customers.